# POC End-to-End Validation

Full pipeline run against one or more video clips with ground-truth comparison.

**Purpose:** Final accuracy check before field deployment.  
**Requires:** A labelled clip in `data/raw_footage/` and matching ground truth in `data/ground_truth/`.

Ground truth format (`data/ground_truth/{test_type}.json`):
```json
{
  "test_type": "explosiveness",
  "clips": {
    "sample_clip.mp4": {
      "students": [
        {"bib": 7,  "metric_value": 34.2, "metric_unit": "cm"},
        {"bib": 14, "metric_value": 28.8, "metric_unit": "cm"}
      ]
    }
  }
}
```

In [ ]:
import sys

from notebooks.poc_validation import VIDEO_PATHS
sys.path.insert(0, '..')

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# Single video or list of videos
# ── Choose one ───────────────────────────────────────────────────────────────
TEST_TYPE = "fitness"    # agility | fitness | shuttle | sprint | explosiveness

# Video map — override VIDEO_PATH below to use a different clip
_VIDEO_MAP = {
    "agility":       "agility_test.mp4",
    "fitness":       "shuttles_test_behind.mp4",
    "shuttle":       "shuttles_test.mp4",
    "sprint":        "sprint_test.mp4",
    "explosiveness": "explosiveness_test.mp4",
}
VIDEO_PATH = Path("../data/raw_footage") / _VIDEO_MAP.get(TEST_TYPE, "shuttles_test.mp4")
VIDEO_PATHS = [VIDEO_PATH]
# Load geometry config
config_file = Path("../configs/test_configs") / f"{TEST_TYPE}.json"
with open(config_file) as f:
    geometry_config = json.load(f)


Videos:       1
  - ../data/raw_footage/shuttles_test_behind.mp4  exists=True
Ground truth: ../data/ground_truth/fitness.json  exists=False


In [ ]:
# ── Load ground truth (all clips) ────────────────────────────────────────────


No ground truth file — will report results without accuracy metrics.


## Calibration: cone_layout (optional)

Use **cone_layout** instead of explicit `cone_world_coords_cm` when you only know spacing and first cone. Add to your test config or override here:

```python
# Linear (shuttle/sprint): cones at 0, 200, 400, ... cm
CONE_LAYOUT = {"pattern": "linear", "first_cone_cm": [0, 0], "spacing_cm": 200, "direction": "x"}

# Grid: rows × cols
CONE_LAYOUT = {"pattern": "grid", "first_cone_cm": [0, 0], "spacing_cm": 150, "rows": 2, "cols": 4}

# Clustered (cones around each person): needs cluster_radius_px + origins per cluster
CONE_LAYOUT = {
    "pattern": "clustered",
    "cluster_radius_px": 150,
    "clusters": [{"origin_cm": [i*250, 0], "spacing_cm": 50, "direction": "x"} for i in range(5)],
}
```

In [ ]:
# ── Run full pipeline (per video) ─────────────────────────────────────────────
import json
from worker.celery_app import _get_extractor
from pipeline.cache import PipelineCache
from pipeline.ingest import extract_frames
from pipeline.detect import PersonDetector
from pipeline.track import PersonTracker
from pipeline.pose import PoseEstimator
from pipeline.ocr import BibOCR, resolve_bibs
from pipeline.calibrate import Calibrator
JOB_ID = f"notebook-eval-{TEST_TYPE}"

cache = PipelineCache(job_id=JOB_ID, cache_root=Path('../data/cache'))
detector = PersonDetector()
estimator = PoseEstimator()
calibrator = Calibrator(
    detector_backend='sam3_prompt',
    sam_model_path='sam3.pt',
    sam_prompt='training cone',
)

video_runs = []
for video_path in VIDEO_PATHS:
    if not video_path.exists():
        print(f"Skipping {video_path.name} (not found)")
        continue

    run = {'path': video_path, 'name': video_path.name}
    frames, frame_indices, timestamps = [], [], []
    for fi, frame, ts in extract_frames(str(video_path), target_fps=15):
        frames.append(frame); frame_indices.append(fi); timestamps.append(ts)
    run['frames'], run['frame_indices'], run['timestamps'] = frames, frame_indices, timestamps
    print(f"[{video_path.name}] [1/6] Ingested {len(frames)} frames")

    all_detections = []
    for i, frame in enumerate(frames):
        dets = detector.detect(frame)
        for d in dets: d.frame_idx = frame_indices[i]
        all_detections.append(dets)
    run['all_detections'] = all_detections
    print(f"[{video_path.name}] [2/6] Detected {sum(len(d) for d in all_detections)} persons")

    tracker = PersonTracker()  # Fresh tracker per video
    all_tracks = []
    for i, (frame, dets) in enumerate(zip(frames, all_detections)):
        all_tracks.append(tracker.update(dets, frame_idx=frame_indices[i]))
    run['all_tracks'] = all_tracks
    print(f"[{video_path.name}] [3/6] Tracked — {len(set(t.track_id for ft in all_tracks for t in ft))} IDs")

    all_poses = []
    for frame, tracks in zip(frames, all_tracks):
        all_poses.append(estimator.estimate_batch(frame, tracks))
    run['all_poses'] = all_poses
    print(f"[{video_path.name}] [4/6] Poses — {sum(len(p) for p in all_poses)} estimates")

    world_coords = geometry_config.get("cone_world_coords_cm", [])
    CONE_LAYOUT = geometry_config.get("cone_layout", {})
    calibration = calibrator.calibrate_from_layout(frames[0], CONE_LAYOUT)
    print(f"[{video_path.name}] [6/6] Calibration — {calibration.method}  px/cm={calibration.pixels_per_cm}")

    video_runs.append(run)

print(f"\nProcessed {len(video_runs)} video(s)")

[shuttles_test_behind.mp4] [1/6] Ingested 444 frames


/home/alex/PycharmProjects/vigour-poc/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:230: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5080 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(
CUDA device not supported by this PyTorch build (e.g. RTX 5080/5090 sm_120). Falling back to CPU for detection.


[shuttles_test_behind.mp4] [2/6] Detected 4031 persons
[shuttles_test_behind.mp4] [3/6] Tracked — 24 IDs


/home/alex/PycharmProjects/vigour-poc/.venv/lib/python3.11/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
Falling back to CPU for pose inference.


Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-aic-coco_pt-aic-coco_420e-256x192-63eb25f7_20230126.pth
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-aic-coco_pt-aic-coco_420e-256x192-63eb25f7_20230126.pth


...

[shuttles_test_behind.mp4] [4/6] Poses — 3988 estimates


/home/alex/PycharmProjects/vigour-poc/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(



Ultralytics 8.4.19 🚀 Python-3.11.12 torch-2.4.0+cu121 CPU (Intel Core i9-9820X 3.30GHz)
0: 1036x1036 39 training cones, 10912.2ms
Speed: 5.6ms preprocess, 10912.2ms inference, 14.2ms postprocess per image at shape (1, 3, 1036, 1036)
Results saved to /home/alex/PycharmProjects/vidoe-detector/runs/segment/predict30


findHomography failed for both normal and reversed order. Points may be degenerate, have too many outliers, or cone layout does not match. Detected 39 cones. Try: detector_backend='hsv', tune HSV ranges, or set reverse_order in layout.


[shuttles_test_behind.mp4] [6/6] Calibration — homography  px/cm=None

Processed 1 video(s)


In [ ]:
# ── Extract metrics (per video) ───────────────────────────────────────────────
configs_dir  = Path('../configs/test_configs')
config_file  = configs_dir / f"{TEST_TYPE}.json"
geometry_cfg = json.loads(config_file.read_text()) if config_file.exists() else {}

for run in video_runs:
    extractor = _get_extractor(TEST_TYPE, geometry_cfg, run['calibration'])
    results = extractor.extract(run['all_tracks'], run['all_poses'], run['frames'])
    run['results'] = results
    cache.save_results(results)

print(f"\n=== {TEST_TYPE.upper()} RESULTS ===")
for run in video_runs:
    print(f"\n--- {run['name']} ---")
    for r in run['results']:
        bib = r.student_bib if r.student_bib and r.student_bib != -1 else f"(track {r.track_id})"
        print(f"  Bib {str(bib):>4s}  {r.metric_value:.2f} {r.metric_unit}  "
              f"conf={r.confidence_score:.2f}  flags={r.flags or '—'}")

Invalid calibration.



=== FITNESS RESULTS ===

--- shuttles_test_behind.mp4 ---


In [ ]:
# ── Accuracy analysis (per video) ─────────────────────────────────────────────
clips_gt = gt_data.get('clips', {})
all_errors = []

for run in video_runs:
    clip_name = run['name']
    students = clips_gt.get(clip_name, {}).get('students', [])
    ground_truth = {s['bib']: s['metric_value'] for s in students}
    results = run['results']

    if not ground_truth:
        print(f"\n[{clip_name}] No ground truth — skipping accuracy.")
        continue

    errors, rows = [], []
    for r in results:
        gt = ground_truth.get(r.student_bib)
        if gt is not None:
            err = r.metric_value - gt
            errors.append(abs(err))
            rows.append((r.student_bib, r.metric_value, gt, err, r.confidence_score))

    if rows:
        all_errors.extend(errors)
        print(f"\n[{clip_name}]")
        print(f"{'Bib':>5} {'Pred':>8} {'GT':>8} {'Error':>8} {'Conf':>6}")
        print("─" * 42)
        for bib, pred, gt, err, conf in sorted(rows):
            flag = "✓" if abs(err) <= 2.0 else "✗"
            print(f"{bib:>5} {pred:>8.2f} {gt:>8.2f} {err:>+8.2f} {conf:>6.2f}  {flag}")
        print(f"MAE: {np.mean(errors):.3f}  Max: {max(errors):.3f}  n={len(errors)}")

        fig, ax = plt.subplots(figsize=(8, 4))
        bibs_r = [r[0] for r in rows]
        pred_v = [r[1] for r in rows]
        gt_v = [r[2] for r in rows]
        x = np.arange(len(rows))
        ax.bar(x - 0.2, pred_v, 0.4, label='Predicted', color='steelblue')
        ax.bar(x + 0.2, gt_v, 0.4, label='Ground Truth', color='tomato')
        ax.set_xticks(x); ax.set_xticklabels([f"Bib {b}" for b in bibs_r])
        ax.set_ylabel(results[0].metric_unit if results else '')
        ax.set_title(f'{TEST_TYPE} — {clip_name} — Predicted vs Ground Truth')
        ax.legend(); plt.tight_layout(); plt.show()
    else:
        print(f"\n[{clip_name}] No overlap between results bibs and ground truth bibs.")

if all_errors:
    print(f"\n=== Overall MAE (all videos): {np.mean(all_errors):.3f}  n={len(all_errors)} ===")


[shuttles_test_behind.mp4] No ground truth — skipping accuracy.


In [ ]:
# ── Save annotated validation videos ──────────────────────────────────────────
from pipeline.visualise import PipelineVisualiser, VisOptions, render_top_down_view

output_dir = Path('../data/annotated')
output_dir.mkdir(parents=True, exist_ok=True)

for run in video_runs:
    stem = run['path'].stem
    out_name = f'poc_validation_{TEST_TYPE}.mp4' if len(video_runs) == 1 else f'poc_validation_{TEST_TYPE}_{stem}.mp4'
    output_path = output_dir / out_name
    opts = VisOptions(
        show_calibration_grid=run['calibration'].is_valid,
        show_top_down_view=run['calibration'].is_valid,
    )
    with PipelineVisualiser(output_path, test_type=TEST_TYPE, fps=15, options=opts) as vis:
        for frame, tracks, poses, ts in zip(
            run['frames'], run['all_tracks'], run['all_poses'], run['timestamps']
        ):
            vis.write_frame(frame, tracks, poses, run['calibration'], run['results'], timestamp_s=ts)
    print(f"Annotated video saved → {output_path}")

Annotated video saved → ../data/annotated/poc_validation_fitness.mp4


In [ ]:
# ── Top-down world-coords view (sample frame) ─────────────────────────────────
# Shows person positions and cone positions in world coordinates (cm)
if video_runs:
    run = video_runs[0]
    # Pick a frame with activity (e.g. middle of video)
    idx = min(len(run['frames']) // 2, len(run['frames']) - 1)
    frame = run['frames'][idx]
    tracks = run['all_tracks'][idx]
    poses = run['all_poses'][idx]
    calib = run['calibration']

    # Standalone top-down view (larger for inspection)
    top_down = render_top_down_view(calib, tracks, poses, size=(400, 400))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(top_down[:, :, ::-1])  # BGR → RGB
    axes[0].set_title('Top-down view (world coords, cm)')
    axes[0].axis('off')

    # Annotated frame with top-down inset
    opts = VisOptions(show_top_down_view=True, top_down_view_size=(280, 280))
    canvas = PipelineVisualiser.annotate_frame(
        frame, tracks, poses, calib, run['results'],
        test_type=TEST_TYPE, opts=opts,
    )
    axes[1].imshow(canvas[:, :, ::-1])
    axes[1].set_title(f'Frame {idx} with top-down inset')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()